# 4.4 缩放定律与训练规划 (Scaling Laws & Training Planning)

> 🕐 预估学习时间：55分钟

缩放定律揭示了模型性能与参数量、数据量、计算量之间的幂律关系，是大模型训练规划的核心理论基础。本节涵盖Kaplan和Chinchilla缩放定律、代码实现、资源规划及涌现能力。

**学习主题：**

1. 缩放定律基础（Kaplan Scaling Laws）
2. Chinchilla缩放定律与计算最优分配
3. 缩放定律的代码实现（ScalingLawModel类）
4. 训练资源规划（TrainingPlanner类）
5. 涌现能力与缩放（EmergentAbility类）

## 1. 缩放定律基础

缩放定律（Scaling Laws）描述了模型性能（通常用损失函数衡量）与模型规模（参数量N、数据量D、计算量C）之间的幂律关系。

**Kaplan缩放定律（2020）：**

- **参数缩放**：L(N) = L₀ + A/N^α，其中L₀为不可约损失
- **数据缩放**：L(D) = L₀ + B/D^β
- **计算缩放**：L(C) = L₀ + G/C^γ

**关键发现：**

1. 损失随规模增加呈幂律下降，可预测地改善
2. 参数量、数据量、计算量各自独立影响损失
3. 大模型在计算效率上更优（Kaplan建议优先增大模型）

**不可约损失L₀**代表自然语言的熵下限，即使无限规模也无法突破。

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 缩放定律基础 (Kaplan Scaling Laws) ===')

# Kaplan缩放定律参数
# L(N) = L0 + A / N^alpha
# L(D) = L0 + B / D^beta
# L(C) = L0 + G / C^gamma
L0 = 1.70
A_N = 40.0
alpha_N = 0.35
B_D = 30.0
beta_D = 0.40
G_C = 20.0
gamma_C = 0.30

# 参数量缩放
param_counts = [1e6, 1e7, 1e8, 1e9, 1e10, 1e11]
fixed_D = 1e10

print('\n--- 参数量缩放 (固定数据量=10B tokens) ---')
header_n = '参数量(N)'
header_l = '损失(L)'
print(f'  {header_n:<15} {header_l:<10}')
for N in param_counts:
    loss = L0 + A_N / (N ** alpha_N) + B_D / (fixed_D ** beta_D)
    print(f'  {N:<15,.0f} {loss:<10.4f}')

# 计算量缩放
compute_budgets = [1e15, 1e16, 1e17, 1e18, 1e19, 1e20]
print('\n--- 计算量缩放 ---')
header_c = '计算量(FLOPs)'
print(f'  {header_c:<18} {header_l:<10}')
for C in compute_budgets:
    loss = L0 + G_C / (C ** gamma_C)
    print(f'  {C:<18,.0f} {loss:<10.4f}')

# 幂律关系验证
print('\n--- 幂律关系验证 ---')
log_C_vals = [math.log10(c) for c in compute_budgets]
log_loss_vals = [math.log10(L0 + G_C / (c ** gamma_C)) for c in compute_budgets]
slope = (log_loss_vals[-1] - log_loss_vals[0]) / (log_C_vals[-1] - log_C_vals[0])
print(f'  log(L) vs log(C) 斜率: {slope:.4f} (理论值: -{gamma_C})')

# 三种缩放关系的对比
print('\n--- 三种缩放关系对比 ---')
N_ref = 1e9
D_ref = 1e10
loss_ref = L0 + A_N / (N_ref ** alpha_N) + B_D / (D_ref ** beta_D)
print(f'  参考模型: N={N_ref:.0e}, D={D_ref:.0e}')
print(f'  参考损失: {loss_ref:.4f}')

scale_factor = 10
loss_10x_N = L0 + A_N / ((N_ref * scale_factor) ** alpha_N) + B_D / (D_ref ** beta_D)
loss_10x_D = L0 + A_N / (N_ref ** alpha_N) + B_D / ((D_ref * scale_factor) ** beta_D)
loss_10x_both = L0 + A_N / ((N_ref * scale_factor) ** alpha_N) + B_D / ((D_ref * scale_factor) ** beta_D)

print(f'  N扩大10倍: 损失={loss_10x_N:.4f} (下降{loss_ref - loss_10x_N:.4f})')
print(f'  D扩大10倍: 损失={loss_10x_D:.4f} (下降{loss_ref - loss_10x_D:.4f})')
print(f'  N和D同时扩大10倍: 损失={loss_10x_both:.4f} (下降{loss_ref - loss_10x_both:.4f})')

print(f'\nKey: Kaplan缩放定律表明模型损失与参数量N、数据量D、计算量C呈幂律关系，损失随规模增加而平滑下降')

## 2. Chinchilla缩放定律

Chinchilla（2022）修正了Kaplan的结论，提出了计算最优分配策略。

**核心发现：**

- 对于给定计算预算C，最优的参数量N和数据量D应**等比例增长**
- 最优token数约为参数量的**20倍**（D/N ≈ 20）
- Kaplan的建议（D/N ≈ 3）导致模型过大、训练不足

**Chinchilla策略：**

- 计算预算 C = 6·N·D（训练FLOPs近似公式）
- 最优分配：N_opt ∝ C^0.5, D_opt ∝ C^0.5
- 在相同计算预算下，Chinchilla策略比Kaplan策略损失更低

**实践意义：** 不要只追求大模型，要确保训练数据量与模型规模匹配。

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== Chinchilla缩放定律 ===')

# Chinchilla损失函数: L(N, D) = A/N^alpha + B/D^beta + L0
A = 40.0
alpha = 0.35
B = 30.0
beta = 0.40
L0 = 1.70

def chinchilla_loss(N, D):
    '''计算Chinchilla损失'''
    return A / (N ** alpha) + B / (D ** beta) + L0

def find_optimal_ND(C, ratio=20):
    '''给定计算预算C, 找到最优的N和D
    C = 6 * N * D, D = ratio * N
    N = sqrt(C / (6 * ratio))
    '''
    N_opt = math.sqrt(C / (6 * ratio))
    D_opt = ratio * N_opt
    return N_opt, D_opt

# 不同计算预算下的最优分配
compute_budgets = [1e18, 1e19, 1e20, 1e21, 1e22, 1e23]
print('\n--- Chinchilla最优分配 ---')
header_c = '计算量(FLOPs)'
header_n = '最优N(参数)'
header_d = '最优D(tokens)'
header_l = '损失'
header_r = 'D/N比'
print(f'  {header_c:<16} {header_n:<16} {header_d:<16} {header_l:<10} {header_r:<8}')
for C in compute_budgets:
    N_opt, D_opt = find_optimal_ND(C)
    loss = chinchilla_loss(N_opt, D_opt)
    ratio = D_opt / N_opt
    print(f'  {C:<16,.0e} {N_opt:<16,.0f} {D_opt:<16,.0f} {loss:<10.4f} {ratio:<8.1f}')

# Kaplan vs Chinchilla对比
print('\n--- Kaplan vs Chinchilla对比 ---')
C_test = 1e21
N_chinchilla, D_chinchilla = find_optimal_ND(C_test)
loss_chinchilla = chinchilla_loss(N_chinchilla, D_chinchilla)

N_kaplan = math.sqrt(C_test / (6 * 3))
D_kaplan = 3 * N_kaplan
loss_kaplan = chinchilla_loss(N_kaplan, D_kaplan)

print(f'  计算预算: {C_test:.0e} FLOPs')
print(f'  Chinchilla: N={N_chinchilla:.2e}, D={D_chinchilla:.2e}, D/N={D_chinchilla/N_chinchilla:.1f}, 损失={loss_chinchilla:.4f}')
print(f'  Kaplan:     N={N_kaplan:.2e}, D={D_kaplan:.2e}, D/N={D_kaplan/N_kaplan:.1f}, 损失={loss_kaplan:.4f}')
improvement = loss_kaplan - loss_chinchilla
print(f'  Chinchilla损失改善: {improvement:.4f} ({improvement/loss_kaplan:.1%})')

# 最优D/N比的敏感性分析
print('\n--- D/N比敏感性分析 ---')
ratios = [5, 10, 15, 20, 25, 30, 50]
header_ratio = 'D/N比'
header_n2 = 'N'
header_d2 = 'D'
header_l2 = '损失'
print(f'  {header_ratio:<10} {header_n2:<15} {header_d2:<15} {header_l2:<10}')
for r in ratios:
    N_opt, D_opt = find_optimal_ND(C_test, ratio=r)
    loss = chinchilla_loss(N_opt, D_opt)
    print(f'  {r:<10} {N_opt:<15,.0f} {D_opt:<15,.0f} {loss:<10.4f}')

print(f'\nKey: Chinchilla发现N和D应等比例缩放，最优token数约为参数量的20倍，比Kaplan策略更计算高效')

## 3. 缩放定律的代码实现

ScalingLawModel类实现缩放定律的拟合和预测功能。

**关键功能：**

- **数据生成**：模拟不同参数量对应的损失数据
- **幂律拟合**：在对数空间使用最小二乘法拟合 L = L₀ + A/N^α
- **性能预测**：根据拟合结果预测更大模型的损失
- **缩放曲线**：展示损失随参数量的变化趋势

**拟合方法：** 将幂律关系转化为对数空间的线性关系：log(L - L₀) = log(A) - α·log(N)，然后使用线性回归求解参数。

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 缩放定律代码实现 ScalingLawModel ===')

class ScalingLawModel:
    '''缩放定律模型：拟合幂律关系，预测更大模型的性能'''

    def __init__(self, L0=1.70):
        self.L0 = L0
        self.A = None
        self.alpha = None
        self.fitted = False

    def generate_scaling_data(self, param_counts):
        '''生成不同参数量对应的损失数据（模拟真实训练）'''
        true_A = 40.0
        true_alpha = 0.35
        losses = []
        for N in param_counts:
            noise = torch.randn(1).item() * 0.01
            loss = self.L0 + true_A / (N ** true_alpha) + noise
            losses.append(loss)
        return losses

    def fit(self, param_counts, losses):
        '''拟合幂律关系: L = L0 + A / N^alpha
        在对数空间线性回归: log(L - L0) = log(A) - alpha * log(N)
        '''
        log_N = [math.log(n) for n in param_counts]
        log_L_minus_L0 = [math.log(max(l - self.L0, 1e-8)) for l in losses]

        n = len(log_N)
        sum_x = sum(log_N)
        sum_y = sum(log_L_minus_L0)
        sum_xy = sum(x * y for x, y in zip(log_N, log_L_minus_L0))
        sum_x2 = sum(x ** 2 for x in log_N)

        slope = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x ** 2)
        intercept = (sum_y - slope * sum_x) / n

        self.alpha = -slope
        self.A = math.exp(intercept)
        self.fitted = True

        print(f'  拟合结果: A={self.A:.4f}, alpha={self.alpha:.4f}')
        print(f'  (真值: A=40.0, alpha=0.35)')

    def predict(self, N):
        '''预测参数量N对应的损失'''
        if not self.fitted:
            raise ValueError('Model not fitted')
        return self.L0 + self.A / (N ** self.alpha)

    def predict_batch(self, param_counts):
        '''批量预测'''
        return [self.predict(N) for N in param_counts]

    def print_scaling_curve(self, param_counts, actual_losses=None):
        '''打印缩放曲线'''
        header_p = '参数量(N)'
        header_pred = '预测损失'
        if actual_losses:
            header_actual = '实际损失'
            header_err = '误差'
            print(f'\n  {header_p:<15} {header_pred:<12} {header_actual:<12} {header_err:<10}')
        else:
            print(f'\n  {header_p:<15} {header_pred:<12}')

        for i, N in enumerate(param_counts):
            pred = self.predict(N)
            if actual_losses:
                actual = actual_losses[i]
                error = abs(pred - actual)
                print(f'  {N:<15,.0f} {pred:<12.4f} {actual:<12.4f} {error:<10.4f}')
            else:
                print(f'  {N:<15,.0f} {pred:<12.4f}')

    def extrapolate(self, target_params):
        '''外推预测更大模型的性能'''
        pred_loss = self.predict(target_params)
        print(f'\n  外推预测: N={target_params:.2e} -> L={pred_loss:.4f}')
        return pred_loss

# 使用小模型数据拟合缩放定律
model = ScalingLawModel(L0=1.70)

# 生成小模型训练数据 (1M ~ 1B)
small_params = [1e6, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9]
actual_losses = model.generate_scaling_data(small_params)

print('--- 拟合缩放定律 ---')
model.fit(small_params, actual_losses)

# 打印拟合曲线对比
model.print_scaling_curve(small_params, actual_losses)

# 外推预测更大模型 (10B, 100B, 1T)
print('\n--- 外推预测大模型 ---')
large_params = [1e10, 1e11, 1e12]
model.print_scaling_curve(large_params)

# 外推预测
for target in [1e10, 1e11, 1e12]:
    model.extrapolate(target)

# 计算损失下降率
print('\n--- 规模翻倍的损失下降 ---')
base_N = 1e9
for multiplier in [2, 4, 8, 16, 32]:
    target_N = base_N * multiplier
    base_loss = model.predict(base_N)
    target_loss = model.predict(target_N)
    drop = base_loss - target_loss
    drop_pct = drop / base_loss
    print(f'  {multiplier}x规模: N={target_N:.0e}, L={target_loss:.4f}, 下降={drop:.4f} ({drop_pct:.1%})')

print(f'\nKey: ScalingLawModel通过对数空间线性回归拟合幂律关系，可以从小模型数据外推预测大模型的性能')

## 4. 训练资源规划

TrainingPlanner类根据目标模型大小估算训练所需的GPU数量、训练时间和成本。

**关键公式：**

- **训练计算量**：C ≈ 6·N·D（6倍参数量乘以数据量）
- **训练时间**：T = C / (n_gpu × FLOPs_per_gpu × MFU)
- **训练成本**：Cost = T × n_gpu × $/gpu/hour

**关键概念：**

- **MFU（Model FLOPs Utilization）**：模型算力利用率，通常40-50%
- **Chinchilla策略**：D = 20·N，确保数据量与模型规模匹配
- **GPU规格**：以A100为例，FP16算力312 TFLOPS，80GB显存

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 训练资源规划 TrainingPlanner ===')

class TrainingPlanner:
    '''训练资源规划：估算GPU数量、训练时间、成本'''

    def __init__(self):
        self.gpu_flops = 312e12  # A100 FP16 算力 (FLOPs/s)
        self.gpu_memory = 80     # GB
        self.gpu_cost = 2.5      # $/GPU/hour
        self.mfu = 0.4           # Model FLOPs Utilization

    def estimate_compute(self, N, D):
        '''估算训练计算量 (FLOPs)
        训练FLOPs约等于 6 * N * D
        '''
        return 6 * N * D

    def estimate_training_time(self, N, D, n_gpus):
        '''估算训练时间 (小时)'''
        total_flops = self.estimate_compute(N, D)
        effective_flops = n_gpus * self.gpu_flops * self.mfu
        time_seconds = total_flops / effective_flops
        return time_seconds / 3600

    def estimate_cost(self, N, D, n_gpus):
        '''估算训练成本 (美元)'''
        hours = self.estimate_training_time(N, D, n_gpus)
        return hours * n_gpus * self.gpu_cost

    def plan_training(self, target_params, tokens_per_param=20):
        '''规划训练方案'''
        D = target_params * tokens_per_param
        total_flops = self.estimate_compute(target_params, D)

        print(f'\n=== 训练规划: {target_params:.2e} 参数模型 ===')
        print(f'  模型参数量: {target_params:.2e}')
        print(f'  训练token数: {D:.2e} (D/N={tokens_per_param})')
        print(f'  总计算量: {total_flops:.2e} FLOPs')

        gpu_options = [64, 128, 256, 512, 1024, 2048]
        header_g = 'GPU数'
        header_t = '训练时间'
        header_d = '天数'
        header_c = '成本($)'
        print(f'\n  {header_g:<10} {header_t:<15} {header_d:<10} {header_c:<15}')

        for n_gpus in gpu_options:
            hours = self.estimate_training_time(target_params, D, n_gpus)
            cost = hours * n_gpus * self.gpu_cost
            days = hours / 24
            print(f'  {n_gpus:<10} {hours:<15.1f}h {days:<10.1f} ${cost:<14,.0f}')

        return total_flops

    def compare_models(self):
        '''对比不同规模模型的训练需求'''
        models = OrderedDict([
            ('1B', 1e9),
            ('7B', 7e9),
            ('13B', 13e9),
            ('70B', 70e9),
            ('175B', 175e9),
        ])

        print('\n=== 不同规模模型训练对比 (256 GPUs) ===')
        header_m = '模型'
        header_p = '参数量'
        header_d = 'Token数'
        header_t = '训练天数'
        header_c = '成本($M)'
        print(f'  {header_m:<8} {header_p:<12} {header_d:<15} {header_t:<12} {header_c:<12}')

        for name, N in models.items():
            D = N * 20
            hours = self.estimate_training_time(N, D, 256)
            cost = hours * 256 * self.gpu_cost
            days = hours / 24
            cost_m = cost / 1e6
            print(f'  {name:<8} {N:<12,.0f} {D:<15,.0f} {days:<12.1f} ${cost_m:<11.2f}M')

    def estimate_memory(self, N, optimizer_factor=8):
        '''估算模型所需显存 (GB)
        optimizer_factor: Adam约8倍参数量 (参数+梯度+优化器状态)
        '''
        params_bytes = N * 2  # FP16, 2 bytes per param
        total_bytes = params_bytes * optimizer_factor
        return total_bytes / 1e9  # 转换为GB

# 创建训练规划器
planner = TrainingPlanner()

# 规划7B模型训练
planner.plan_training(7e9, tokens_per_param=20)

# 规划70B模型训练
planner.plan_training(70e9, tokens_per_param=20)

# 对比不同规模模型
planner.compare_models()

# 显存估算
print('\n=== 模型显存估算 ===')
header_m = '模型'
header_p = '参数量'
header_mem = '显存(GB)'
header_g = '最少GPU数'
print(f'  {header_m:<8} {header_p:<12} {header_mem:<12} {header_g:<12}')
model_sizes = OrderedDict([
    ('1B', 1e9),
    ('7B', 7e9),
    ('13B', 13e9),
    ('70B', 70e9),
    ('175B', 175e9),
])
for name, N in model_sizes.items():
    mem = planner.estimate_memory(N)
    min_gpus = math.ceil(mem / planner.gpu_memory)
    print(f'  {name:<8} {N:<12,.0f} {mem:<12.1f} {min_gpus:<12}')

print(f'\nKey: TrainingPlanner基于6*N*D公式估算计算量，结合GPU算力和MFU预测训练时间与成本，辅助训练资源决策')

## 5. 涌现能力与缩放

涌现能力（Emergent Abilities）是指模型在规模达到一定临界点后突然出现的能力。

**涌现能力的特征：**

- **非连续性**：能力在临界点前接近随机水平，超过后急剧提升
- **规模依赖**：不同能力有不同的涌现阈值
- **不可预测性**：难以从小模型外推大模型的能力

**典型涌现能力：**

- 算术运算（~10B参数）
- 多步推理（~50B参数）
- 代码生成（~100B参数）

**争议：** 部分研究认为涌现能力可能是评估指标的非线性造成的假象，使用连续指标时能力提升可能是平滑的。

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 涌现能力与缩放 EmergentAbility ===')

class EmergentAbility:
    '''涌现能力：模拟能力随规模涌现的现象'''

    def __init__(self):
        self.abilities = OrderedDict()

    def define_ability(self, name, threshold, sharpness=10):
        '''定义一个涌现能力
        threshold: 涌现的参数量阈值
        sharpness: 涌现的急剧程度
        '''
        self.abilities[name] = OrderedDict([
            ('threshold', threshold),
            ('sharpness', sharpness),
        ])

    def compute_ability_score(self, name, model_size):
        '''计算给定模型规模下的能力得分
        使用sigmoid函数模拟涌现:
        小于阈值时接近0, 超过阈值后急剧上升趋近1
        '''
        ability = self.abilities[name]
        threshold = ability['threshold']
        sharpness = ability['sharpness']
        log_ratio = math.log10(model_size / threshold)
        score = 1 / (1 + math.exp(-sharpness * log_ratio))
        return score

    def simulate_emergence(self, model_sizes):
        '''模拟所有能力在不同规模下的涌现'''
        header_size = '模型参数量'
        line = f'  {header_size:<15}'
        for name in self.abilities:
            line += f' {name:<14}'
        print(line)

        for size in model_sizes:
            line = f'  {size:<15,.0f}'
            for name in self.abilities:
                score = self.compute_ability_score(name, size)
                line += f' {score:<14.4f}'
            print(line)

    def find_emergence_point(self, name, model_sizes):
        '''找到能力涌现的临界点 (得分超过0.5的规模)'''
        for size in model_sizes:
            score = self.compute_ability_score(name, size)
            if score >= 0.5:
                return size
        return None

    def demonstrate_emergence(self):
        '''演示涌现能力现象'''
        # 定义几种涌现能力
        self.define_ability('算术运算', 1e10, sharpness=8)
        self.define_ability('逻辑推理', 5e10, sharpness=10)
        self.define_ability('代码生成', 1e11, sharpness=12)
        self.define_ability('多语言翻译', 2e10, sharpness=6)

        # 模型规模范围
        model_sizes = [1e8, 5e8, 1e9, 5e9, 1e10, 5e10, 1e11, 5e11, 1e12]

        print('\n--- 涌现能力模拟 ---')
        self.simulate_emergence(model_sizes)

        # 找到各能力的涌现点
        print('\n--- 涌现临界点 (得分>=0.5) ---')
        for name in self.abilities:
            point = self.find_emergence_point(name, model_sizes)
            if point:
                print(f'  {name}: 涌现于 {point:.2e} 参数')
            else:
                print(f'  {name}: 未涌现')

        # 涌现能力的特征
        print('\n--- 涌现能力特征 ---')
        print('  1. 非连续性: 能力在临界点前接近随机, 超过后急剧提升')
        print('  2. 规模依赖: 不同能力有不同的涌现阈值')
        print('  3. 不可预测: 无法从小模型外推大模型的能力')

    def compare_emergence_vs_smooth(self):
        '''对比涌现式提升与平滑提升'''
        print('\n--- 涌现式 vs 平滑式能力提升 ---')
        model_sizes = [1e8, 5e8, 1e9, 5e9, 1e10, 5e10, 1e11, 5e11, 1e12]
        header_n = '参数量'
        header_e = '涌现式'
        header_s = '平滑式'
        print(f'  {header_n:<15} {header_e:<12} {header_s:<12}')

        threshold = 1e10
        for size in model_sizes:
            # 涌现式: sigmoid (急剧)
            log_ratio = math.log10(size / threshold)
            emergent = 1 / (1 + math.exp(-10 * log_ratio))
            # 平滑式: 幂律 (平滑)
            smooth = min(1.0, (size / threshold) ** 0.3)
            print(f'  {size:<15,.0f} {emergent:<12.4f} {smooth:<12.4f}')

        print('\n  涌现式: 使用sigmoid函数, 临界点前后变化剧烈')
        print('  平滑式: 使用幂律函数, 能力随规模平滑增长')

# 演示涌现能力
emergence = EmergentAbility()
emergence.demonstrate_emergence()
emergence.compare_emergence_vs_smooth()

print(f'\nKey: 涌现能力表现为能力在规模临界点前接近随机、超过后急剧提升，不同能力有不同的涌现阈值')

## 📝 课后思考题

1. Kaplan和Chinchilla缩放定律的核心区别是什么？为什么Chinchilla建议D/N≈20而不是更大的模型？
2. 如果你有固定的计算预算（如1e22 FLOPs），如何根据缩放定律确定最优的模型参数量和训练数据量？
3. 涌现能力是否真的是非连续的？评估指标的选择如何影响我们对涌现的判断？
4. 在实际训练规划中，除了参数量N和数据量D，还有哪些因素会影响训练时间和成本？